# DeepCSI on real COST2100 data — Colab GPU runner

Trains the DeepCSI autoencoder at CR = 4 / 16 / 32 on the **COST2100** CSI feedback
benchmark (the dataset released with CsiNet), evaluates it against a DCT baseline,
and compares to published results.

**Before anything else:** `Runtime → Change runtime type → T4 GPU → Save`.

The dataset download happens on Google's network inside this notebook — nothing
lands on your laptop. At the end you download ~25 MB of artifacts for the local
FastAPI + Streamlit demo.

Published references on this benchmark (indoor):

| CR | CsiNet | CRNet |
|---:|-------:|------:|
| 4  | -17.36 dB | -26.99 dB |
| 16 |  -8.65 dB | -11.35 dB |
| 32 |  -6.24 dB |  -8.93 dB |

Landing near these means the numbers are real. Anything near **-38 dB** means a
metric is being computed on offset data again — see `REAL_DATA.md`.

> **Colab sessions die** after ~90 min idle (4 h max) and take the filesystem with
> them. Cell 5b saves the prepared tensors to your Drive so a restart costs you
> minutes, not another 2.5 GB download. Don't skip it.

## Cell 1 — Confirm the GPU is actually attached

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), \
    "No GPU. Runtime > Change runtime type > T4 GPU, then Runtime > Restart and run all."

## Cell 2 — Get the DeepCSI code

The `karim/real-data` branch is local-only (gh is not authenticated), so upload
the zip. On your laptop it is already built at:

```
C:\Users\bigbo\Downloads\deepcsi-karim.zip
```

Run the cell, click **Choose Files**, pick that zip.

In [ ]:
from google.colab import files
import zipfile, os

up = files.upload()                     # pick deepcsi-karim.zip
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall(".")

os.chdir("DeepCSI-karim")
print("cwd:", os.getcwd())
!ls

*Alternative, if you later push the branch somewhere you can reach:*

```python
!git clone --branch karim/real-data https://github.com/<you>/DeepCSI.git DeepCSI-karim
%cd DeepCSI-karim
```

## Cell 3 — Install the two packages Colab lacks

In [ ]:
!pip install -q gdown h5py
import scipy, pandas, h5py, gdown
print("ready")

## Cell 4 — Get COST2100

We need three **indoor** files (~2.5 GB total):

| file | contents |
|---|---|
| `DATA_Htrainin.mat` | 100,000 training samples |
| `DATA_Hvalin.mat` | 30,000 validation samples |
| `DATA_Htestin.mat` | 20,000 test samples |

`DATA_HtestFin_all.mat` (1.3 GB) is **not needed** — it only exists for the
frequency-domain rho calculation, which we don't reproduce.

Try **Route A** first. If Google Drive returns a quota error
("Too many users have viewed or downloaded this file recently"), use **Route B**,
which is immune to it and is also the better long-term setup.

### Route A — direct download (try this first, ~2-4 min)

In [ ]:
import os, glob, shutil
os.makedirs("data/raw/COST2100", exist_ok=True)

# Folder published by the CRNet authors (same files as the CsiNet release).
FOLDER = "https://drive.google.com/drive/folders/1_lAMLk_5k1Z8zJQlTr5NRnSD6ACaNRtj"
!gdown --folder {FOLDER} -O data/raw/COST2100 --remaining-ok

# gdown may nest things a level deep; flatten so the .mat files sit directly here.
for f in glob.glob("data/raw/COST2100/**/*.mat", recursive=True):
    dest = os.path.join("data/raw/COST2100", os.path.basename(f))
    if os.path.abspath(f) != os.path.abspath(dest):
        shutil.move(f, dest)

for f in sorted(glob.glob("data/raw/COST2100/*.mat")):
    print(f"{os.path.getsize(f)/1e6:8.0f} MB  {os.path.basename(f)}")

### Route B — via your own Google Drive (immune to quota, survives restarts)

Do this **once**, in your browser:

1. Open the [COST2100 folder](https://drive.google.com/drive/folders/1_lAMLk_5k1Z8zJQlTr5NRnSD6ACaNRtj)
2. Right-click the folder → **Organise** → **Add shortcut to Drive** → **My Drive**

That is a pointer, not a copy — it is instant and uses none of your storage quota.
Then run the cell below. After this, every future session reads straight from
Drive with no download at all.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob, os, shutil
# Adjust if you renamed the shortcut, or if it sits inside a subfolder.
DRIVE_DIR = "/content/drive/MyDrive/COST2100"

os.makedirs("data/raw/COST2100", exist_ok=True)
found = glob.glob(f"{DRIVE_DIR}/**/DATA_H*in.mat", recursive=True)
assert found, f"No indoor .mat files under {DRIVE_DIR}. Check the shortcut name:\n" \
              + "\n".join(sorted(os.listdir("/content/drive/MyDrive"))[:40])

# Copy locally: FUSE-mounted Drive is slow for the repeated reads scipy does.
for f in found:
    dest = os.path.join("data/raw/COST2100", os.path.basename(f))
    if not os.path.exists(dest):
        print("copying", os.path.basename(f), "...")
        shutil.copy(f, dest)

for f in sorted(glob.glob("data/raw/COST2100/*.mat")):
    print(f"{os.path.getsize(f)/1e6:8.0f} MB  {os.path.basename(f)}")

### Route C — Dropbox fallback

Only if both routes above fail. This pulls the whole folder as one zip, including
the outdoor and `_all` files you don't need, so it is the slowest option.

In [ ]:
!wget -q --show-progress -O cost2100.zip \
  "https://www.dropbox.com/scl/fo/tqhriijik2p76j7kfp9jl/h?rlkey=4r1zvjpv4lh5h4fpt7lbpus8c&dl=1"
!unzip -o -j cost2100.zip "*DATA_H*in.mat" -d data/raw/COST2100
!ls -la data/raw/COST2100

## Cell 5 — Prepare the tensors

Converts the `(N, 2048)` MATLAB arrays into the `(N, 2, 32, 32)` float32 tensors
the pipeline consumes, and writes `norm_params.json` recording the CsiNet offset
convention (0.5 is the complex zero point).

The train/val/test split ships with the dataset, so there is no re-splitting and
no normalisation leakage across the split boundary.

`--subsample 20000` keeps the run quick. Drop the flag to train on all 100,000.

In [ ]:
!python data/prepare_cost2100.py \
    --mat-dir data/raw/COST2100 \
    --output-dir data/processed_cost2100 \
    --environment indoor \
    --subsample 20000

### Cell 5b — Back the prepared tensors up to Drive

~200 MB. If the session dies, Cell 5c restores in seconds instead of re-downloading
2.5 GB. Skip only if you mounted Drive in Route B and are feeling lucky.

In [ ]:
from google.colab import drive
import os, shutil
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

BACKUP = "/content/drive/MyDrive/deepcsi_prepared"
os.makedirs(BACKUP, exist_ok=True)
for f in ("train.npy", "val.npy", "test.npy", "norm_params.json"):
    shutil.copy(f"data/processed_cost2100/{f}", f"{BACKUP}/{f}")
print("backed up to", BACKUP)
!ls -la {BACKUP}

In [ ]:
# Cell 5c -- RESTORE after a session restart. Skips Cells 4 and 5 entirely.
from google.colab import drive
import os, shutil
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

BACKUP = "/content/drive/MyDrive/deepcsi_prepared"
os.makedirs("data/processed_cost2100", exist_ok=True)
for f in ("train.npy", "val.npy", "test.npy", "norm_params.json"):
    shutil.copy(f"{BACKUP}/{f}", f"data/processed_cost2100/{f}")
print("restored")
!ls -la data/processed_cost2100

## Cell 6 — Train

**CR=4 first.** It is the easiest configuration and acts as the gate: if it does
not get near the CsiNet reference (-17 dB), stop and debug rather than burning
time on the other two.

A few minutes each on a T4.

In [ ]:
COMMON = "--data-dir data/processed_cost2100 --batch-size 200 --epochs 40 --patience 10 --amp"
!python models/train.py --compression-ratio 4 {COMMON}

In [ ]:
!python models/train.py --compression-ratio 16 {COMMON}

In [ ]:
!python models/train.py --compression-ratio 32 {COMMON}

## Cell 7 — Evaluate

Writes `results/metrics.csv`, the comparison pivots, and the figures. NMSE is
printed in the log-of-mean convention on the de-offset channel — the same
convention CsiNet and CRNet use, so the columns are directly comparable.

In [ ]:
!python models/evaluate.py \
    --data-dir data/processed_cost2100 \
    --weights-dir models/weights \
    --results-dir results

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv("results/metrics.csv"))
for fig in ["nmse_vs_cr.png", "beamforming_gain_vs_cr.png", "reconstruction_heatmaps.png"]:
    try:
        display(Image(f"results/figures/{fig}"))
    except Exception as e:
        print(fig, "->", e)

## Cell 8 — Sanity check before you trust any of this

Three checks. If any fails, the result is not presentable — read `REAL_DATA.md`
for what each failure means.

In [ ]:
import pandas as pd

df = pd.read_csv("results/metrics.csv")
deep = df[df.method == "DeepCSI"].sort_values("compression_ratio")
col = "nmse_db_aggregate" if "nmse_db_aggregate" in deep else "nmse_db_mean"
ok = True

# 1. NMSE must degrade as compression increases. A flat curve means the
#    bottleneck was never the binding constraint.
nmse = deep[col].tolist()
mono = all(a < b for a, b in zip(nmse, nmse[1:]))
print(f"1. NMSE degrades with CR : {'PASS' if mono else 'FAIL'}  {nmse}")
ok &= mono

# 2. Nothing near -38 dB. That was the signature of measuring NMSE on
#    DC-offset data, which inflates the denominator by ~35 dB.
sane = all(n > -30 for n in nmse)
print(f"2. No implausible -38 dB : {'PASS' if sane else 'FAIL'}")
ok &= sane

# 3. rho must vary across CR. Pinned near 99.98% means the de-offset was skipped.
gains = deep["beamforming_gain_mean"].tolist()
varies = (max(gains) - min(gains)) > 1e-3
print(f"3. rho varies across CR  : {'PASS' if varies else 'FAIL'}  {[round(g,4) for g in gains]}")
ok &= varies

print()
print("ALL CHECKS PASSED" if ok else "SOMETHING IS WRONG -- do not present these numbers")

In [ ]:
# Side-by-side with the published baselines.
import pandas as pd

df = pd.read_csv("results/metrics.csv")
deep = df[df.method == "DeepCSI"].set_index("compression_ratio")
col = "nmse_db_aggregate" if "nmse_db_aggregate" in deep else "nmse_db_mean"

table = pd.DataFrame({
    "DeepCSI (ours)": deep[col],
    "CsiNet (paper)": pd.Series({4: -17.36, 16: -8.65, 32: -6.24}),
    "CRNet (paper)":  pd.Series({4: -26.99, 16: -11.35, 32: -8.93}),
})
table["vs CsiNet"] = (table["DeepCSI (ours)"] - table["CsiNet (paper)"]).round(2)
print(table.round(2).to_string())
print("\nNegative 'vs CsiNet' means we beat the paper at that CR.")

## Cell 9 — Download the artifacts

Everything the local demo needs. The test set is trimmed to 2,000 samples to keep
the download small — that is the range the dashboard's sample selector uses.

In [ ]:
import numpy as np, shutil, os, json

shutil.rmtree("export", ignore_errors=True)
os.makedirs("export/data", exist_ok=True)
os.makedirs("export/models/weights", exist_ok=True)

test = np.load("data/processed_cost2100/test.npy")
np.save("export/data/test.npy", test[:2000])
shutil.copy("data/processed_cost2100/norm_params.json", "export/data/norm_params.json")

# norm_params records the full split size; correct it for the trimmed copy.
p = json.load(open("export/data/norm_params.json"))
p["num_test"] = int(min(2000, len(test)))
p["note"] = "test.npy trimmed to the first 2000 samples for the local demo"
json.dump(p, open("export/data/norm_params.json", "w"), indent=2)

for cr in (4, 16, 32):
    src = f"models/weights/deepcsi_cr{cr}.pt"
    if os.path.exists(src):
        shutil.copy(src, f"export/models/weights/deepcsi_cr{cr}.pt")
shutil.copytree("results", "export/results", dirs_exist_ok=True)

!cd export && zip -qr ../deepcsi_artifacts.zip .
print(f"deepcsi_artifacts.zip -> {os.path.getsize('deepcsi_artifacts.zip')/1e6:.1f} MB")
!unzip -l deepcsi_artifacts.zip

In [ ]:
from google.colab import files
files.download("deepcsi_artifacts.zip")

# Also drop a copy in Drive, in case the browser download is interrupted.
import shutil, os
if os.path.ismount("/content/drive"):
    shutil.copy("deepcsi_artifacts.zip", "/content/drive/MyDrive/deepcsi_artifacts.zip")
    print("copy saved to Drive")

## Cell 10 — Back on your laptop

```powershell
cd C:\Users\bigbo\Downloads\DeepCSI\DeepCSI-karim

# unpack next to the repo
Expand-Archive -Force $HOME\Downloads\deepcsi_artifacts.zip .
mkdir -Force data\processed_cost2100
Move-Item -Force data\test.npy, data\norm_params.json data\processed_cost2100\

$env:DATA_DIR = "data/processed_cost2100"
python preflight.py                      # must end: DEEPCSI PREFLIGHT: PASS
uvicorn backend.app:app --reload --port 8000
```

Then in a second terminal:

```powershell
cd C:\Users\bigbo\Downloads\DeepCSI\DeepCSI-karim
streamlit run frontend/app.py
```